# Desafio 3 - ZettaLab

## Análise Exploratória de Dados (EDA)

### Importação de Bibliotecas

In [1]:
import pandas as pd             # Biblioteca para manipulação e análise de dados
import numpy as np              # Biblioteca para calculo de vetores e matrizes
import sys                      # Biblioteca para acessar variaveis e funções que interagem fortemente com o interpretador
import os                       # Biblioteca para interação com arquivos e diretórios a nivel sistema operacional
import basedosdados as bd       # Biblioteca para acessar o datalake público do site BasedosDados
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
from datetime import datetime, timedelta    # 
import requests                 # 

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import config_path          # Módulo que salva todos os caminhos de diretórios utilizados no projeto

from scripts import features        # Módulo de criação e manipulação de features
from scripts import modeling        # Módulo de modelagem de IA
from scripts import utils           # Módulo de utilitários genéricos
from scripts import pre_processing  # Módulo de pré-processamento e obtenção de dados.

from dotenv import load_dotenv      # Biblioteca para carregar variáveis de ambiente de arquivos .env
load_dotenv()
NASA_FIRMS_API_KEY = os.getenv("NASA_FIRMS_API_KEY") # Coloca a API KEY da NASA FIRMS numa constante

In [2]:
df_uf_cods = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_municipios.csv")
df_mun_cods = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_ufs.csv")
df_mun_area = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "municipio_area_2024.csv")
df_uf_area = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "uf_area_2024.csv")
df_rg_area = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "rg_area_2024.csv")
df_inmet_2025 = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "df_inmet_2025.csv")
df_dados_BDMEP = pd.read_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "dados_meteorologicos_BDMEP_parquet.parquet")

In [3]:
df_inmet_2025.head()

,DATA,ESTACAO,UF,LATITUDE,LONGITUDE,CODIGO_WMO,precip_mm,temp_max,temp_min,umid_max,umid_min,vento
0,2025-01-01 00:00:00,BRASILIA,DF,-15.789444,-47.925833,A001,NaN,NaN,NaN,92.0,90.0,1.8
1,2025-01-01 01:00:00,BRASILIA,DF,-15.789444,-47.925833,A001,NaN,NaN,NaN,93.0,92.0,1.9
2,2025-01-01 02:00:00,BRASILIA,DF,-15.789444,-47.925833,A001,NaN,NaN,NaN,92.0,91.0,1.4
3,2025-01-01 03:00:00,BRASILIA,DF,-15.789444,-47.925833,A001,NaN,NaN,NaN,92.0,91.0,1.1
4,2025-01-01 04:00:00,BRASILIA,DF,-15.789444,-47.925833,A001,NaN,NaN,NaN,95.0,92.0,0.3


In [4]:
df_dados_BDMEP.head()

,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo
0,2024,10,2024-10-30 04:21:00,Cerrado,TO,Tocantins,1715002,Nova Rosalândia,-10.50309,-49.01587,NOAA-21,0.0,24.09,0.00,1.9
1,2024,10,2024-10-30 04:21:00,Cerrado,TO,Tocantins,1717800,Ponte Alta do Bom Jesus,-11.88486,-46.53584,NOAA-21,2.0,2.24,0.03,1.5
2,2024,10,2024-10-30 04:21:00,Cerrado,MA,Maranhão,2102804,Carolina,-7.46682,-47.18346,NOAA-21,0.0,8.18,0.04,1.8
3,2024,10,2024-10-30 04:21:00,Cerrado,PI,Piauí,2201101,Avelino Lopes,-10.14042,-44.02159,NOAA-21,9.0,0.41,1.00,1.1
4,2024,10,2024-10-30 04:21:00,Caatinga,PE,Pernambuco,2612604,Santa Maria da Boa Vista,-8.74347,-39.68242,NOAA-21,12.0,0.00,1.00,3.2


In [ ]:
# O foco do trabalho são em incêndios no norte de minas gerais

### Geração de Gráficos

#### NASA FIRMS

In [6]:


# --- 1. Baixar os dados do FIRMS ---
# Exemplo: MODIS Active Fire Data (últimos 7 dias) para o Brasil
url = f"https://firms.modaps.eosdis.nasa.gov/api/csv/active_fire/c6/country/BR?apiKey={NASA_FIRMS_API_KEY}"

response = requests.get(url)
if response.status_code != 200:
    print(response.status_code)
    raise Exception("Erro ao baixar dados do FIRMS. Verifique sua API key e URL.")

# Ler CSV em pandas
data = StringIO(response.text)
df = pd.read_csv(data)

# --- 2. Pré-processamento ---
# Converter data para datetime
df['acq_date'] = pd.to_datetime(df['acq_date'])

# Contar número de focos por dia
fires_per_day = df.groupby('acq_date').size().reset_index(name='num_focos')

# --- 3. Plotar gráfico ---
plt.figure(figsize=(12,6))
sns.lineplot(data=fires_per_day, x='acq_date', y='num_focos', marker='o')
plt.title('Número de focos de queimadas no Brasil (últimos dias)')
plt.xlabel('Data')
plt.ylabel('Número de focos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

400


Exception: Erro ao baixar dados do FIRMS. Verifique sua API key e URL.